# Model Evaluation & Baseline Comparison

This notebook loads the pre-trained XGBoost models and evaluates them against the Persistence and Climatology baselines on the test set. It reports both Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE)[cite: 1].

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import json
from pathlib import Path

DATA_PATH = "clean_jun_v2.csv"
MODEL_DIR = Path("models_final")

In [2]:
# 1. Load Metadata
with open(MODEL_DIR / 'metadata.json', 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
horizon_names = metadata['horizon_names']
target_cols = metadata['target_cols']
OPERATIONAL_START = int(metadata['operational_window'].split(':')[0])
OPERATIONAL_END = int(metadata['operational_window'].split('-')[1].split(':')[0])
steps = list(range(1, 7))

print(f"Metadata loaded. Horizons to evaluate: {horizon_names}")

Metadata loaded. Horizons to evaluate: ['30m', '60m', '90m', '120m', '150m', '180m']


In [3]:
# 2. Recreate Data Pipeline for Test Set Extraction
df_raw = pd.read_csv(DATA_PATH)
df_raw['TIMESTAMP'] = pd.to_datetime(df_raw['TIMESTAMP'])
df_raw = df_raw.set_index('TIMESTAMP').sort_index()

def engineer_features(df, resample_rule='30min'):
    df = df.copy()
    df = df.resample(resample_rule).mean()
    
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    df['doy_sin']  = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
    df['doy_cos']  = np.cos(2 * np.pi * df.index.dayofyear / 365.25)

    df['v_solar'] = df['solar_kw'].diff()
    df['a_solar'] = df['v_solar'].diff()
    df['solar_volatility_2h'] = df['solar_kw'].rolling(window=4).std()
    
    if 'rh_avg' in df.columns:
        df['v_humidity'] = df['rh_avg'].diff()
        df['humidity_solar_risk'] = (df['rh_avg'] / 100) * (1 - df['solar_kw'])

    physical_sensors = ['solar_kw', 'net_rad_wm2', 'temp_c', 'rh_avg', 'wind_speed_ms', 'wind_dir', 'rain_mm']
    lag_cols = [c for c in physical_sensors if c in df.columns]
    for col in lag_cols:
        df[f'{col}_lag_6h'] = df[col].shift(12)
        df[f'{col}_lag_24h'] = df[col].shift(48)
        
    for i in range(1, 7):
        df[f'solar_lag_{i*30}m'] = df['solar_kw'].shift(i)

    df['solar_time_ratio'] = df['solar_kw'] / (df['hour_sin'] + 0.1)
    df['solar_time_ratio'] = df['solar_time_ratio'].clip(0, 5)
    
    wd_rad = np.radians(df['wind_dir'])
    df['wind_x'] = df['wind_speed_ms'] * np.cos(wd_rad)
    df['wind_y'] = df['wind_speed_ms'] * np.sin(wd_rad)

    if 'leaf_wetness' in df.columns:
        df['wet_ground_lag_6h'] = df['leaf_wetness'].shift(12)

    df['solar_trend_1h'] = df['solar_kw'] - df['solar_kw'].shift(2)
    return df.dropna()

df = engineer_features(df_raw)

df_day = df.between_time(f'{OPERATIONAL_START}:00', f'{OPERATIONAL_END}:00').copy()
for s in steps:
    df_day[f'target_solar_diff_{s*30}m'] = df_day['solar_kw'].shift(-s) - df_day['solar_kw']
df_day = df_day.dropna()

train_end = int(len(df_day) * 0.8)
train = df_day.iloc[:train_end]
test  = df_day.iloc[train_end:]

print(f"Test set prepared: {len(test)} rows")

Test set prepared: 14850 rows


In [4]:
# 3. Load XGBoost Models
models_day = {}
for h in horizon_names:
    m = xgb.XGBRegressor()
    m.load_model(MODEL_DIR / f"daylight_{h}.json")
    models_day[f"daylight_{h}"] = (m, h)

print("Successfully loaded all 6 XGBoost horizon models.")

Successfully loaded all 6 XGBoost horizon models.


In [5]:
# 4. Define Evaluation Functions with MAE and RMSE
def evaluate_models(models_dict, dataset):
    rows = []
    solar_vals = dataset['solar_kw'].values
    for label, (m, h) in models_dict.items():
        target = f'target_solar_diff_{h}'
        actual_target = dataset[target].values
        pred_ramp = m.predict(dataset[feature_cols])
        pred_abs = np.clip(solar_vals + pred_ramp, 0, None)
        actual_abs = solar_vals + actual_target
        
        mask = (~np.isnan(actual_abs)) & (actual_abs > 0.01)
        if mask.sum() == 0: continue
        
        mae_w = mean_absolute_error(actual_abs[mask], pred_abs[mask]) * 1000
        rmse_w = np.sqrt(mean_squared_error(actual_abs[mask], pred_abs[mask])) * 1000
        
        rows.append({'model': 'xgboost', 'horizon': h, 'horizon_min': int(h.replace('m','')),
                     'mae_wm2': round(mae_w, 2), 'rmse_wm2': round(rmse_w, 2)})
    return pd.DataFrame(rows).sort_values('horizon_min')

def evaluate_persistence(dataset):
    rows = []
    solar_vals = dataset['solar_kw'].values
    for s, h in zip(steps, horizon_names):
        target = f'target_solar_diff_{h}'
        actual_abs = solar_vals + dataset[target].values
        pred_abs = solar_vals
        
        mask = (~np.isnan(actual_abs)) & (actual_abs > 0.01)
        if mask.sum() == 0: continue
            
        mae_w = mean_absolute_error(actual_abs[mask], pred_abs[mask]) * 1000
        rmse_w = np.sqrt(mean_squared_error(actual_abs[mask], pred_abs[mask])) * 1000
        
        rows.append({'baseline': 'persistence', 'horizon': h, 'horizon_min': s*30,
                     'mae_wm2': round(mae_w, 2), 'rmse_wm2': round(rmse_w, 2)})
    return pd.DataFrame(rows).sort_values('horizon_min')

def evaluate_climatology(dataset, train_set):
    clim = train_set.groupby([train_set.index.hour, train_set.index.minute])['solar_kw'].mean()
    rows = []
    solar_vals = dataset['solar_kw'].values
    for s, h in zip(steps, horizon_names):
        target = f'target_solar_diff_{h}'
        actual_abs = solar_vals + dataset[target].values
        
        future_times = dataset.index + pd.Timedelta(minutes=s*30)
        clim_vals = np.array([clim.get((t.hour, t.minute), 0) for t in future_times])
        
        mask = (~np.isnan(actual_abs)) & (actual_abs > 0.01)
        if mask.sum() == 0: continue
            
        mae_w = mean_absolute_error(actual_abs[mask], clim_vals[mask]) * 1000
        rmse_w = np.sqrt(mean_squared_error(actual_abs[mask], clim_vals[mask])) * 1000
        
        rows.append({'baseline': 'climatology', 'horizon': h, 'horizon_min': s*30,
                     'mae_wm2': round(mae_w, 2), 'rmse_wm2': round(rmse_w, 2)})
    return pd.DataFrame(rows).sort_values('horizon_min')

In [7]:
# 5. Run Evaluations & Merge Final Table
eval_xgb = evaluate_models(models_day, test)
eval_persist = evaluate_persistence(test)
eval_clim = evaluate_climatology(test, train)

summary = eval_xgb[['horizon', 'mae_wm2', 'rmse_wm2']].copy()
summary.columns = ['horizon', 'xgb_mae_wm2', 'xgb_rmse_wm2']

for df_baseline, name in [(eval_persist, 'persistence'), (eval_clim, 'climatology')]:
    summary = summary.merge(
        df_baseline[['horizon', 'mae_wm2', 'rmse_wm2']].rename(
            columns={'mae_wm2': f'{name}_mae_wm2', 'rmse_wm2': f'{name}_rmse_wm2'}),
        on='horizon'
    )

print("Final Comparison: XGBoost vs Baselines (MAE and RMSE in W/m²)")
print(summary)

Final Comparison: XGBoost vs Baselines (MAE and RMSE in W/m²)
  horizon  xgb_mae_wm2  xgb_rmse_wm2  persistence_mae_wm2  \
0     30m        34.57         52.45                44.70   
1     60m        45.77         67.60                73.80   
2     90m        50.21         72.67                99.12   
3    120m        56.56         80.53               121.14   
4    150m        58.23         82.74               139.07   
5    180m        61.74         86.52               153.06   

   persistence_rmse_wm2  climatology_mae_wm2  climatology_rmse_wm2  
0                 62.97                80.85                105.71  
1                 97.39                81.43                105.80  
2                125.24                82.82                106.36  
3                149.60                85.22                107.88  
4                170.51                88.99                111.12  
5                186.98                94.02                116.49  
